# Lesson 15: Enterprise OAuth 2.1, S256 PKCE & Identity Hardening

In this notebook, you will:
1. Generate an ephemeral RS256 RSA keypair with `cryptography`.
2. Sign and verify MCP Bearer JWTs using `PyJWT` with `iss`, `aud`, and `scope` checks.
3. Test `/.well-known/oauth-protected-resource` discovery in `FastAPI`.

In [ ]:
!pip install -q -U fastapi PyJWT cryptography httpx

In [ ]:
import time
import jwt
from cryptography.hazmat.primitives.asymmetric import rsa

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

ISSUER = "https://idp.example.com/"
AUDIENCE = "https://mcp.example.com/mcp"

token = jwt.encode(
    {"iss": ISSUER, "aud": AUDIENCE, "sub": "alice", "scope": "mcp:read mcp:write", "exp": int(time.time()) + 300},
    private_key,
    algorithm="RS256",
)

claims = jwt.decode(token, public_key, algorithms=["RS256"], audience=AUDIENCE, issuer=ISSUER)
print("RS256 JWT verified! Subject:", claims["sub"], "Scopes:", claims["scope"])

### Challenge Exercise
Create a `FastAPI` endpoint `/.well-known/oauth-protected-resource` returning `{"resource": AUDIENCE, "authorization_servers": [ISSUER], "scopes_supported": ["mcp:read", "mcp:write"]}` and verify it with `TestClient`.